---
title: "Data Cleaning"
format:
    html: 
        code-fold: false
---

{{< include instructions.qmd >}} 


{{< include overview.qmd >}} 

{{< include methods.qmd >}} 

# Code 

Provide the source code used for this section of the project here.

If you're using a package for code organization, you can import it at this point. However, make sure that the **actual workflow steps**—including data processing, analysis, and other key tasks—are conducted and clearly demonstrated on this page. The goal is to show the technical flow of your project, highlighting how the code is executed to achieve your results.

Ensure that the code is well-commented to enhance readability and understanding for others who may review or use it. If relevant, link to additional documentation or external references that explain any complex components. This section should give readers a clear view of how the project is implemented from a technical perspective.

This page is a technical narrative, NOT just a notebook with a collection of code cells, include in-line Prose, to describe what is going on.

### I collected 2,812 movies from TMDB with complete budget and revenue data. First, I'm loading the data to see what I'm working with.

In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('../../data/raw-data/tmdb_movies_raw.csv')

# basic exploration
print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())
print("\nFirst few rows:")
df.head()

Dataset shape: (2812, 18)

Column names:
['movie_id', 'title', 'release_date', 'budget', 'revenue', 'runtime', 'vote_average', 'vote_count', 'popularity', 'overview', 'genres', 'primary_genre', 'production_companies', 'cast', 'cast_popularity', 'director', 'release_year', 'release_month']

First few rows:


,movie_id,title,release_date,budget,revenue,runtime,vote_average,vote_count,popularity,overview,genres,primary_genre,production_companies,cast,cast_popularity,director,release_year,release_month
0,1084242,Zootopia 2,2025-11-26,150000000,642000000,107,7.725,344,512.6445,After cracking the biggest case in Zootopia's ...,Animation|Family|Comedy|Adventure,Animation,Walt Disney Animation Studios,Ginnifer Goodwin|Jason Bateman|Ke Huy Quan|For...,5.07572,Jared Bush,2025,11
1,533533,TRON: Ares,2025-10-08,220000000,142073994,119,6.489,539,458.1744,A highly sophisticated Program called Ares is ...,Science Fiction|Adventure|Action,Science Fiction,Walt Disney Pictures|Sean Bailey Productions,Jared Leto|Greta Lee|Evan Peters|Gillian Ander...,4.80276,Joachim Rønning,2025,10
2,1246049,Dracula,2025-07-30,52000000,25788866,130,7.100,518,248.9951,When a 15th-century prince denounces God after...,Horror|Fantasy|Romance,Horror,EuropaCorp|Luc Besson Production|TF1 Films Pro...,Caleb Landry Jones|Zoë Bleu Sidel|Christoph Wa...,3.63440,Luc Besson,2025,7
3,1242898,Predator: Badlands,2025-11-05,105000000,175186920,107,7.379,548,190.6551,"Cast out from his clan, a young Predator finds...",Action|Science Fiction|Adventure,Action,20th Century Studios|Lawrence Gordon Productio...,Elle Fanning|Dimitrius Schuster-Koloamatangi|R...,6.45618,Dan Trachtenberg,2025,11
4,967941,Wicked: For Good,2025-11-19,150000000,642000000,137,6.700,366,142.4678,As an angry mob rises against the Wicked Witch...,Fantasy|Adventure|Romance,Fantasy,Universal Pictures|Marc Platt Productions|dentsu,Ariana Grande|Cynthia Erivo|Jonathan Bailey|Je...,6.13622,Jon M. Chu,2025,11


### Now I'm checking to see if there are any missing values or data quality issues that I'll need to address during cleaning.

In [9]:
print("Missing Values by Column:")
print("="*60)
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Percentage': missing_pct
})
print(missing_df[missing_df['Missing Count'] > 0])

print("\n" + "="*60)
print("Budget Statistics:")
print(df['budget'].describe())

print("\n" + "="*60)
print("Revenue Statistics:")
print(df['revenue'].describe())

print("\n" + "="*60)
print("Runtime Statistics:")
print(df['runtime'].describe())

print("\n" + "="*60)
print("Movies by Year (last 10 years):")
print(df['release_year'].value_counts().sort_index().tail(10))

Missing Values by Column:
                      Missing Count  Percentage
production_companies              1    0.035562
cast                              1    0.035562

Budget Statistics:
count    2.812000e+03
mean     4.536952e+07
std      5.817536e+07
min      5.000000e+00
25%      8.289067e+06
50%      2.500000e+07
75%      6.000000e+07
max      5.839000e+08
Name: budget, dtype: float64

Revenue Statistics:
count    2.812000e+03
mean     1.590962e+08
std      2.650542e+08
min      1.000000e+00
25%      1.155852e+07
50%      5.614315e+07
75%      1.788447e+08
max      2.923706e+09
Name: revenue, dtype: float64

Runtime Statistics:
count    2812.000000
mean      112.612731
std        22.772921
min        56.000000
25%        97.000000
50%       108.000000
75%       124.000000
max       254.000000
Name: runtime, dtype: float64

Movies by Year (last 10 years):
release_year
2016     63
2017     74
2018     82
2019     85
2020     40
2021     81
2022     74
2023     85
2024    113
2025 

### Before cleaning, I'm creating the key features I'll need for my analysis: ROI and a success label to predict whether movies are profitable.

In [10]:
# create ROI 
df['roi'] = df['revenue'] / df['budget']

# create profitability label
df['profitable'] = (df['revenue'] > df['budget']).astype(int)

# create success categories
def categorize_success(row):
    roi = row['roi']
    if roi < 1:
        return 'Bomb'  # lost money
    elif roi < 2:
        return 'Modest'  # made money but not much
    elif roi < 5:
        return 'Hit'  # strong performer
    else:
        return 'Blockbuster'  # massive success

df['success_category'] = df.apply(categorize_success, axis=1)

# create budget tiers
def categorize_budget(budget):
    if budget < 20_000_000:
        return 'Low'
    elif budget < 80_000_000:
        return 'Medium'
    elif budget < 150_000_000:
        return 'High'
    else:
        return 'Blockbuster'

df['budget_tier'] = df['budget'].apply(categorize_budget)

# create release season
def get_season(month):
    if month in [6, 7, 8]:
        return 'Summer'
    elif month in [11, 12]:
        return 'Holiday'
    elif month in [3, 4, 5]:
        return 'Spring'
    else:
        return 'Fall'

df['release_season'] = df['release_month'].apply(get_season)

print("New Features Created:")
print(f"ROI range: {df['roi'].min():.2f} to {df['roi'].max():.2f}")
print(f"\nSuccess Distribution:")
print(df['success_category'].value_counts())
print(f"\nProfitable: {df['profitable'].sum()} ({df['profitable'].mean()*100:.1f}%)")
print(f"\nBudget Tiers:")
print(df['budget_tier'].value_counts())
print(f"\nRelease Seasons:")
print(df['release_season'].value_counts())

New Features Created:
ROI range: 0.00 to 12890.39

Success Distribution:
success_category
Hit            936
Blockbuster    706
Bomb           690
Modest         480
Name: count, dtype: int64

Profitable: 2118 (75.3%)

Budget Tiers:
budget_tier
Low            1229
Medium         1060
High            299
Blockbuster     224
Name: count, dtype: int64

Release Seasons:
release_season
Fall       911
Summer     711
Spring     649
Holiday    541
Name: count, dtype: int64


### I noticed some extreme ROI values that are likely data errors (movies with budgets under $1000 or revenues under $1000). I'm filtering these out to ensure data quality.

In [11]:
print("Before cleaning:")
print(f"Total movies: {len(df)}")
print(f"Min budget: ${df['budget'].min():,.0f}")
print(f"Min revenue: ${df['revenue'].min():,.0f}")

# remove unrealistic budget/revenue values
df_clean = df[
    (df['budget'] >= 1_000_000) &  # at least $1M budget
    (df['revenue'] >= 1_000) &      # at least $1k revenue
    (df['roi'] < 100)                # ROI less than 100x 
].copy()

# drop rows with missing cast or production companies
df_clean = df_clean.dropna(subset=['cast', 'production_companies'])

# recalculate features after cleaning
df_clean['profitable'] = (df_clean['revenue'] > df_clean['budget']).astype(int)

print("\nAfter cleaning:")
print(f"Total movies: {len(df_clean)}")
print(f"Movies removed: {len(df) - len(df_clean)}")
print(f"Min budget: ${df_clean['budget'].min():,.0f}")
print(f"Min revenue: ${df_clean['revenue'].min():,.0f}")
print(f"Max ROI: {df_clean['roi'].max():.2f}")
print(f"\nNew Success Distribution:")
print(df_clean['success_category'].value_counts())
print(f"\nNew Profitable Rate: {df_clean['profitable'].mean()*100:.1f}%")

Before cleaning:
Total movies: 2812
Min budget: $5
Min revenue: $1

After cleaning:
Total movies: 2691
Movies removed: 121
Min budget: $1,000,000
Min revenue: $1,361
Max ROI: 94.50

New Success Distribution:
success_category
Hit            913
Bomb           668
Blockbuster    642
Modest         468
Name: count, dtype: int64

New Profitable Rate: 75.1%


### Now that I've engineered features and removed outliers, I'm saving the cleaned dataset for use in my exploratory analysis and modeling phases.

In [12]:
# save cleaned data
output_path = '../../data/processed-data/tmdb_movies_clean.csv'
df_clean.to_csv(output_path, index=False)

print(f"Clean data saved to: {output_path}")
print(f"\nFinal Dataset Summary:")
print(f"Total movies: {len(df_clean):,}")
print(f"Features: {len(df_clean.columns)}")
print(f"Date range: {df_clean['release_year'].min()}-{df_clean['release_year'].max()}")
print(f"Budget range: ${df_clean['budget'].min():,.0f} - ${df_clean['budget'].max():,.0f}")
print(f"Profitable rate: {df_clean['profitable'].mean()*100:.1f}%")

print("\nAll columns:")
print(df_clean.columns.tolist())

# show sample
print("\nSample of cleaned data:")
df_clean[['title', 'budget', 'revenue', 'roi', 'success_category', 'budget_tier']].head(10)

Clean data saved to: ../../data/processed-data/tmdb_movies_clean.csv

Final Dataset Summary:
Total movies: 2,691
Features: 23
Date range: 1934-2025
Budget range: $1,000,000 - $583,900,000
Profitable rate: 75.1%

All columns:
['movie_id', 'title', 'release_date', 'budget', 'revenue', 'runtime', 'vote_average', 'vote_count', 'popularity', 'overview', 'genres', 'primary_genre', 'production_companies', 'cast', 'cast_popularity', 'director', 'release_year', 'release_month', 'roi', 'profitable', 'success_category', 'budget_tier', 'release_season']

Sample of cleaned data:


,title,budget,revenue,roi,success_category,budget_tier
0,Zootopia 2,150000000,642000000,4.280000,Hit,Blockbuster
1,TRON: Ares,220000000,142073994,0.645791,Bomb,Blockbuster
2,Dracula,52000000,25788866,0.495940,Bomb,Medium
3,Predator: Badlands,105000000,175186920,1.668447,Modest,High
4,Wicked: For Good,150000000,642000000,4.280000,Hit,Blockbuster
5,Frankenstein,120000000,480678,0.004006,Bomb,High
6,Zootopia,150000000,1025521689,6.836811,Blockbuster,Blockbuster
7,Bugonia,50000000,34817177,0.696344,Bomb,Medium
8,Demon Slayer: Kimetsu no Yaiba Infinity Castle,20000000,740400000,37.020000,Blockbuster,Medium
9,Now You See Me: Now You Don't,90000000,159000154,1.766668,Modest,High


{{< include closing.qmd >}} 